In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



In [2]:
from microtorch import Tensor

import time
import numpy as np

from microtorch import Tensor, nn
from microtorch.optim import Adam
from microtorch.utils import load_mnist, batch_iterator, evaluate_classifier

import torch
import torch.nn as torch_nn
import torch.optim as torch_optim


np.random.seed(42)
X_train, y_train, X_test, y_test = load_mnist()

In [3]:
def make_microtorch_model():
    return nn.Sequential(
        nn.Linear(784, 128),
        nn.ReLU(),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, 10),
    )


In [4]:
def train_microtorch(epochs=5, batch_size=64, lr=1e-3):
    model = make_microtorch_model()
    criterion = nn.CrossEntropyLoss()
    optimizer = Adam(model.parameters(), lr=lr)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_acc": [],
        "epoch_time": [],
    }

    for epoch in range(epochs):
        start = time.perf_counter()

        epoch_loss = 0.0
        correct = 0
        total = 0

        for X_batch, y_batch in batch_iterator(X_train, y_train, batch_size=batch_size, shuffle=True):
            x = Tensor(X_batch, requires_grad=False)
            logits = model(x)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += float(loss.data) * len(X_batch)
            preds = np.argmax(logits.data, axis=1)
            correct += np.sum(preds == y_batch)
            total += len(X_batch)

        epoch_time = time.perf_counter() - start
        train_loss = epoch_loss / total
        train_acc = correct / total
        test_acc = evaluate_classifier(model, X_test, y_test, batch_size=batch_size)

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        history["epoch_time"].append(epoch_time)

        print(
            f"[MicroTorch] epoch {epoch+1}: "
            f"loss={train_loss:.4f}, "
            f"train_acc={train_acc:.4f}, "
            f"test_acc={test_acc:.4f}, "
            f"time={epoch_time:.2f}s"
        )

    return model, history


In [5]:
class TorchMLP(torch_nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch_nn.Sequential(
            torch_nn.Linear(784, 128),
            torch_nn.ReLU(),
            torch_nn.Linear(128, 64),
            torch_nn.ReLU(),
            torch_nn.Linear(64, 10),
        )

    def forward(self, x):
        return self.net(x)


In [6]:
def train_pytorch(epochs=5, batch_size=64, lr=1e-3):
    torch.manual_seed(42)

    model = TorchMLP()
    criterion = torch_nn.CrossEntropyLoss()
    optimizer = torch_optim.Adam(model.parameters(), lr=lr)

    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    history = {
        "train_loss": [],
        "train_acc": [],
        "test_acc": [],
        "epoch_time": [],
    }

    for epoch in range(epochs):
        start = time.perf_counter()

        perm = torch.randperm(len(X_train_t))
        X_train_epoch = X_train_t[perm]
        y_train_epoch = y_train_t[perm]

        epoch_loss = 0.0
        correct = 0
        total = 0

        model.train()
        for i in range(0, len(X_train_epoch), batch_size):
            X_batch = X_train_epoch[i:i + batch_size]
            y_batch = y_train_epoch[i:i + batch_size]

            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * len(X_batch)
            preds = logits.argmax(dim=1)
            correct += (preds == y_batch).sum().item()
            total += len(X_batch)

        epoch_time = time.perf_counter() - start
        train_loss = epoch_loss / total
        train_acc = correct / total

        model.eval()
        with torch.no_grad():
            logits = model(X_test_t)
            preds = logits.argmax(dim=1)
            test_acc = (preds == y_test_t).float().mean().item()

        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)
        history["epoch_time"].append(epoch_time)

        print(
            f"[PyTorch] epoch {epoch+1}: "
            f"loss={train_loss:.4f}, "
            f"train_acc={train_acc:.4f}, "
            f"test_acc={test_acc:.4f}, "
            f"time={epoch_time:.2f}s"
        )

    return model, history


In [7]:
if __name__ == "__main__":
    _, micro_history = train_microtorch()
    _, torch_history = train_pytorch()

    print("\nFinal Comparison")
    print(
        f"MicroTorch: test_acc={micro_history['test_acc'][-1]:.4f}, "
        f"avg_epoch_time={np.mean(micro_history['epoch_time']):.2f}s"
    )
    print(
        f"PyTorch:    test_acc={torch_history['test_acc'][-1]:.4f}, "
        f"avg_epoch_time={np.mean(torch_history['epoch_time']):.2f}s"
    )


AxisError: axis 1 is out of bounds for array of dimension 1